1. Detect face
2. Crop face
3. Blur check
4. Extract keypoints
5. Estimate pitch & yaw
6. If |yaw| < 25 and |pitch| < 20:
       Align face
       Extract embedding
       Duplicate check
       Save

7. add actual embeddding model
8. use deque voting based approaach for recognition


In [1]:
import cv2
from ultralytics import YOLO
model = YOLO(model="../yolov8n-face.pt")

In [2]:
img  = cv2.imread("../IMG-20230622-WA0001.jpg")
result  = model.track(img,persist=True)
print(result[0].keypoints)


0: 640x480 3 faces, 102.9ms
Speed: 53.8ms preprocess, 102.9ms inference, 8.6ms postprocess per image at shape (1, 3, 640, 480)
ultralytics.engine.results.Keypoints object with attributes:

conf: tensor([[0.8945, 0.8984, 0.9083, 0.8992, 0.9043],
        [0.9084, 0.9017, 0.9100, 0.9095, 0.9046],
        [0.8920, 0.8908, 0.9111, 0.9142, 0.9141]])
data: tensor([[[294.2668, 289.5856,   0.8945],
         [339.6555, 291.3375,   0.8984],
         [320.0507, 312.0952,   0.9083],
         [296.1803, 334.1114,   0.8992],
         [334.3396, 335.7030,   0.9043]],

        [[451.2921, 217.5205,   0.9084],
         [493.3534, 214.4365,   0.9017],
         [473.0557, 239.5414,   0.9100],
         [459.1155, 256.7396,   0.9095],
         [493.5503, 254.2813,   0.9046]],

        [[626.2495, 287.8726,   0.8920],
         [667.6915, 284.8977,   0.8908],
         [646.1226, 309.6871,   0.9111],
         [632.2707, 328.4428,   0.9142],
         [666.3076, 325.9485,   0.9141]]])
has_visible: True
orig_sha

In [5]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


CUDA Available: False
GPU Name: No GPU


In [ ]:
from ultralytics import YOLO
import cv2
from head_pose import estimate_head_pose
from frame_test import blur_score
model = YOLO("../yolov8n-face.pt")
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame,persist=True)


    boxes = results[0].boxes
    keypoints = results[0].keypoints.xy
    if boxes.id is not None:
        for i,(box,track_id) in enumerate(zip(boxes,boxes.id)):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            id = int(track_id)
            face_crop = frame[y1:y2, x1:x2]
            blur = blur_score(face_crop)

            #
            kpts = keypoints[i]
            pitch,yaw,roll=estimate_head_pose(kpts,frame)
            

            face_crop = cv2.resize(face_crop, (112, 112))
            cv2.rectangle(frame,(x1,y1),(x2,y2),color=(0,255,0),)
            cv2.putText(frame,f"{blur}",(x1, y1-10),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            # cv2.imshow("Face Crop", face_crop)

    cv2.imshow("Frame", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 (no detections), 130.1ms
Speed: 2.5ms preprocess, 130.1ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 81.2ms
Speed: 4.1ms preprocess, 81.2ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 63.1ms
Speed: 2.4ms preprocess, 63.1ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 59.9ms
Speed: 2.3ms preprocess, 59.9ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 60.0ms
Speed: 2.5ms preprocess, 60.0ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 48.8ms
Speed: 1.1ms preprocess, 48.8ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 face, 43.7ms
Speed: 1.2ms preprocess, 43.7ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 face, 41.2ms
Speed: 1.6ms preprocess, 41.2ms inference, 0.6ms 

In [ ]:
from frame_test import filter_frame  # import your function
from ultralytics import YOLO
import cv2
from head_pose import estimate_head_pose
from frame_test import blur_score
import time

import numpy as np
from qdrant_utils import FaceDB
model = YOLO("../yolov8n-face.pt")
cap = cv2.VideoCapture(0)
start_time  = time.time()

candidates = []
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame,persist=True)


    boxes = results[0].boxes
    keypoints = results[0].keypoints.xy
    if boxes.id is not None:
        for i,(box,track_id) in enumerate(zip(boxes,boxes.id)):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            id = int(track_id)
            face_crop = frame[y1:y2, x1:x2]

            kpts = keypoints[i]
            # extract face rotation
            pitch,yaw,roll=estimate_head_pose(kpts,frame)
            blur  = blur_score(face_crop)   # get blur score

            # filtering frame based on yaw,pitch and blur score and face alignment using roll
            candidates = filter_frame(
                candidates,
                yaw,
                pitch,
                face_crop,
                roll=roll
            )
            


            face_crop = cv2.resize(face_crop, (112, 112))
            cv2.rectangle(frame,(x1,y1),(x2,y2),color=(0,255,0),)
            cv2.putText(frame,f" {blur}",(x1, y1-10),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            # cv2.imshow("Face Crop", face_crop)

    cv2.imshow("Frame", frame)
    if time.time()-start_time >10:
        break
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

from frame_test import select_candidates

selected, selected_embs = select_candidates(candidates)

print("selected_embs:", selected_embs)
print("len:", len(selected_embs))

if len(selected_embs) > 0:
    print("type first:", type(selected_embs[0]))
    print("shape first:", np.array(selected_embs[0]).shape)
print("yaw:", yaw, "pitch:", pitch)
print("blur:", blur)

for i,img in enumerate(selected):
    cv2.imwrite(f'image{i}.png',img)
db = FaceDB()

# after your select_candidates()
db.delete_by_name("keyur")
db.add_embeddings("keyur", selected_embs)
db.close()


0: 480x640 1 face, 108.5ms
Speed: 2.1ms preprocess, 108.5ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)
[[    0.98338    -0.18047    0.019652]
 [   -0.18148    -0.97466     0.13076]
 [ -0.0044431    -0.13215    -0.99122]]

0: 480x640 1 face, 44.0ms
Speed: 1.3ms preprocess, 44.0ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)
[[    0.97628    -0.18378     0.11446]
 [   -0.19436    -0.97686      0.0893]
 [   0.095402    -0.10943    -0.98941]]

0: 480x640 1 face, 44.2ms
Speed: 1.8ms preprocess, 44.2ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)
[[    0.98273    -0.17722    0.053198]
 [    -0.1794    -0.98298    0.039546]
 [   0.045285   -0.048407     -0.9978]]

0: 480x640 1 face, 45.0ms
Speed: 0.9ms preprocess, 45.0ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)
[[    0.97716    -0.17645     0.11842]
 [   -0.18853    -0.97698     0.09988]
 [   0.098067    -0.11992    -0.98793]]

0: 480x640 1 face, 44.3ms
Spe

In [3]:
db.close()

In [ ]:
from ultralytics import YOLO
import cv2
from head_pose import estimate_head_pose
from frame_test import blur_score
from qdrant_utils import FaceDB
from recog import RecognitionSystem

db = FaceDB()
recog = RecognitionSystem(db)

model = YOLO("../yolov8n-face.pt")
cap = cv2.VideoCapture(0)

track_labels = {}
frame_count =0
confirmed_labels = {}
while True:
    ret, frame = cap.read()

    if not ret:
        break
    frame_count +=1
    results = model.track(frame, persist=True)

    boxes = results[0].boxes
    keypoints = results[0].keypoints.xy
    if boxes.id is not None:
        for i, (box, track_id) in enumerate(zip(boxes, boxes.id)):

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            track_id = int(track_id)

            face_crop = frame[y1:y2, x1:x2]

            kpts = keypoints[i]
            pitch, yaw, roll = estimate_head_pose(kpts, frame)
            # label = recog.process(track_id, yaw, pitch, face_crop, roll)
#####################################################
            if track_id not in confirmed_labels and frame_count%4==0:
                label = recog.process(track_id, yaw, pitch, face_crop, roll)

                if label is not None:
                    key = (track_id, label)

                    track_labels[key] = track_labels.get(key, 0) + 1

                    if track_labels[key] >= 10:
                        confirmed_labels[track_id] = label

            display_name = confirmed_labels.get(track_id, "Unknown")
           ##################### ###############
            # display_name  = "UnKnown"
            # if label is not None:
            #     track_labels[(track_id,label)]   = track_labels.get((track_id,label),0)+1
            #     if track_labels[(track_id,label)] >=10:
            #         display_name = label
            # display_name = track_labels.get(track_id, "Collecting...")

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, display_name, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    cv2.imshow("Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 face, 116.8ms
Speed: 1.7ms preprocess, 116.8ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)
[[    0.99335      0.1004   -0.056325]
 [  -0.080636     0.25763    -0.96287]
 [  -0.082161     0.96101     0.26401]]

0: 480x640 1 face, 66.6ms
Speed: 2.6ms preprocess, 66.6ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)
[[    0.96109     0.19116    -0.19941]
 [   -0.18196    -0.10504    -0.97768]
 [   -0.20784     0.97592   -0.066173]]

0: 480x640 1 face, 55.4ms
Speed: 1.1ms preprocess, 55.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
[[    0.99421   -0.091169   -0.056851]
 [   0.023277    -0.33379     0.94236]
 [   -0.10489    -0.93823    -0.32974]]

0: 480x640 1 face, 50.7ms
Speed: 1.0ms preprocess, 50.7ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)
[[    0.99453   -0.090526     -0.0521]
 [   0.017103    -0.35094     0.93624]
 [   -0.10304    -0.93201    -0.34747]]

0: 480x640 1 face, 52.8ms
Spe

In [2]:
deleted = db.delete_by_name("keyur")
print(deleted)

keyur not exists!!!


# extracting pitch and yaw

In [9]:
import numpy as np
import cv2
kpts = result[0].keypoints.xy[0].cpu().numpy()
left_eye = kpts[0]
right_eye = kpts[1]
nose  = kpts[2]
left_mouth =kpts[3]
right_mouth = kpts[4]

In [ ]:
model_points = np.array([   (0.0, 0.0, 0.0),          # Nose
    (-30.0, 30.0, -30.0),     # Left Eye
    (30.0, 30.0, -30.0),      # Right Eye
    (-25.0, -30.0, -30.0),    # Left Mouth
    (25.0, -30.0, -30.0) ],dtype='double')
image_points = np.array([nose,left_eye,right_eye,left_mouth,right_mouth],dtype='double')

In [14]:
h, w = frame.shape[:2]

focal_length = w
center = (w/2, h/2)

camera_matrix = np.array([
    [focal_length, 0, center[0]],
    [0, focal_length, center[1]],
    [0, 0, 1]
], dtype="double")

NameError: name 'frame' is not defined

In [ ]:
success, rotation_vector, translation_vector = cv2.solvePnP(
    model_points,
    image_points,
    camera_matrix,
    None,
    flags=cv2.SOLVEPNP_ITERATIVE
)

In [ ]:
rotation_matrix, _ = cv2.Rodrigues(rotation_vector)

angles, _, _, _, _, _ = cv2.RQDecomp3x3(rotation_matrix)

pitch = angles[0]
yaw = angles[1]
roll = angles[2]

print("Yaw:", yaw)
print("Pitch:", pitch)